## Screen end members

Screen the end members using a rocksalt template

In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [2]:
from ase.io import read
from mace.calculators import mace_mp
from ase.constraints import FixSymmetry
from ase.filters import FrechetCellFilter
from spglib import find_primitive
from ase import Atoms
from ase.optimize import FIRE
from aiida_vasp.workchains.v2 import VaspBandUpdater, VaspRelaxUpdater,VaspHybridBandUpdater

from aiida import orm

In [3]:
from aiida_grouppathx import GroupPathX
basepath = GroupPathX('hc-zincblend')

In [4]:
def get_primitive_atoms(base):
    cell, pos, numbers = find_primitive((base.cell, base.get_scaled_positions(), base.numbers), 1e-2)
    return Atoms(cell=cell, scaled_positions=pos, numbers=numbers, pbc=True)
base = get_primitive_atoms(read('CdTe.cif'))
base.set_cell(base.cell * 1.1, scale_atoms=True)

In [5]:
def generate_rocksalt(a, b):
    atoms = base.copy()
    atoms.symbols[atoms.symbols == 'Cd'] = 'He'
    atoms.symbols[atoms.symbols == 'Te'] = 'Xe'
    atoms.symbols[atoms.symbols == 'He'] = a
    atoms.symbols[atoms.symbols == 'Xe'] = b
    return atoms

In [6]:
def get_zb(a, b):
    atoms = generate_rocksalt(a, b)
    
    atoms.calc = mace_mp('medium')
    opt = FIRE(FrechetCellFilter(atoms))
    
    opt.run()
    atoms.calc = None
    return atoms

In [7]:
structpath = basepath['structures']
structpath.get_or_create_group()

(<Group: 'hc-zincblend/structures' [type core], of user bzhu@bit.edu.cn>, True)

In [10]:
for A in ['In', 'Ga']:
    for B in ['Sb']:
        atoms = get_zb(A, B)
        structure = orm.StructureData(ase=atoms)
        structure.label = f'{A}{B} INIT'
        structpath.add_node(structure.store(), structure.label.replace(' ', '_').lower())

Using Materials Project MACE for MACECalculator with /home/bonan/.cache/mace/20231203mace128L1_epoch199model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Default dtype float32 does not match model dtype float64, converting models to float32.
      Step     Time          Energy          fmax
FIRE:    0 23:50:44       -6.592115        1.696512
FIRE:    1 23:50:44       -6.675242        1.569189
FIRE:    2 23:50:44       -6.816406        1.309078
FIRE:    3 23:50:44       -6.968532        0.894946
FIRE:    4 23:50:44       -7.066218        0.268128
FIRE:    5 23:50:44       -7.043318        0.561965
FIRE:    6 23:50:44       -7.045640        0.539673
FIRE:    7 23:50:44       -7.049920        0.496260
FIRE:    8 23:50:44       -7.055493        0.433963
FIRE:    9 23:50:44       -7.061509        0.355876
FIRE:   10 23:50:44       -7.067072        0.265743
FIRE:   11 23:50:44       -7.071382        0.167641
F

In [11]:
def launch_hse06(node, label):
    """Generate process builder"""
    queue_name = 'xhhctdnormal'
    code = 'vasp-6.4.2@sugon-xh-v2'
    upd = VaspHybridBandUpdater().apply_preset(structure=node, overrides={
        'lhfcalc': True,
        'hfscreen': 0.2,
        'precfock': 'fast',
        'ispin': 1,
        'gga': None,  # PBE functional
        'magmom': None,
        'ncore': 2,
        'kpar': 16,
    },
         code=code, label=f'{node.get_formula()} {node.label}THSE06')
    upd.set_resources(num_machines=4, tot_num_mpiprocs=64)
    upd.set_options(max_wallclock_seconds=3600 * 48, queue_name=queue_name)
 
    # Only re-try twice if there is convergence problem
    upd.builder.scf.max_iterations = 2

    # Do relaxation - this is needed for getting the IBZKPT
    upd_relax = VaspRelaxUpdater(builder=upd.builder.relax)
    upd_relax.apply_preset(structure=node, overrides={
        #'metagga': 'mbj',
        'ispin': 1,
        'gga': None,
        'magmom': None,
        'ncore': 8,
    },
         code=code, label=f'{node.get_formula()} {node.label} PBE RELAX')
    upd_relax.set_resources(num_machines=1, tot_num_mpiprocs=32)
    upd_relax.set_options(max_wallclock_seconds=3600 * 12, queue_name=queue_name)
    #upd_relax.set_relax_settings(perform=False) # Not actually relaxing
    upd.set_band_settings(band_mode='bradcrack', line_density=15, kpoints_per_split=200)
    running = upd.submit()
    return running, label

In [12]:
def launch_hse06_soc(node, label):
    """Generate process builder"""
    queue_name = 'xhhctdnormal'
    code = 'vasp-6.4.2-ncl@sugon-xh-v2'
    upd = VaspHybridBandUpdater().apply_preset(structure=node, overrides={
        'lhfcalc': True,
        'hfscreen': 0.2,
        'precfock': 'fast',
        'ispin': None, # Use default
        'gga': None,  # PBE functional
        'magmom': None,
        'ncore': 2,
        'kpar': 16,
        'lsorbit': True,
    },
         code=code, label=f'{node.get_formula()} {node.label} HSE06 SOC')
    upd.set_resources(num_machines=4, tot_num_mpiprocs=64)
    upd.set_options(max_wallclock_seconds=3600 * 48, queue_name=queue_name)
 
    # Only re-try twice if there is convergence problem
    upd.builder.scf.max_iterations = 2

    # Do relaxation - this is needed for getting the IBZKPT
    upd_relax = VaspRelaxUpdater(builder=upd.builder.relax)
    upd_relax.apply_preset(structure=node, overrides={
        #'metagga': 'mbj',
        'ispin': 1,
        'gga': None,
        'magmom': None,
        'ncore': 8,
    },
         code=code, label=f'{node.get_formula()} {node.label} PBE RELAX')
    upd_relax.set_resources(num_machines=1, tot_num_mpiprocs=32)
    upd_relax.set_options(max_wallclock_seconds=3600 * 12, queue_name=queue_name)
    #upd_relax.set_relax_settings(perform=False) # Not actually relaxing
    upd.set_band_settings(band_mode='bradcrack', line_density=15, kpoints_per_split=200)
    running = upd.submit()
    return running, label

In [13]:
def launch_mbj_soc(node, label, soc=False):
    """Generate process builder"""
    queue_name = 'xhhctdnormal'
    code = 'vasp-6.4.2-ncl@sugon-xh-v2' if soc else 'vasp-6.4.2@sugon-xh-v2'
    worklabel = f'{node.get_formula()} {node.label} MBJ'
    if soc:
        worklabel += ' SOC'
    upd = VaspHybridBandUpdater().apply_preset(structure=node, overrides={
        'ispin': 1, # Use default
        'metagga': 'mbj',
        'gga': None,
        'ncore': 2,
        'kpar': 16,
        'lsorbit': True,
    },
         code=code, label=worklabel)
    upd.set_resources(num_machines=1, tot_num_mpiprocs=64)
    upd.set_options(max_wallclock_seconds=3600 * 48, queue_name=queue_name)
 
    # Only re-try twice if there is convergence problem
    upd.builder.scf.max_iterations = 2

    # Do relaxation - this is needed for getting the IBZKPT
    upd_relax = VaspRelaxUpdater(builder=upd.builder.relax)
    upd_relax.apply_preset(structure=node, overrides={
        #'metagga': 'mbj',
        'ispin': 1,
        'gga': None,
        'magmom': None,
        'ncore': 8,
    },
         code=code, label=worklabel + 'RELAX')
    upd_relax.set_resources(num_machines=1, tot_num_mpiprocs=32)
    upd_relax.set_options(max_wallclock_seconds=3600 * 12, queue_name=queue_name)
    #upd_relax.set_relax_settings(perform=False) # Not actually relaxing
    upd.set_band_settings(band_mode='bradcrack', line_density=15, kpoints_per_split=200)
    running = upd.submit()
    return running, label

## Analyse results

In [18]:
structpath.show_tree()

structures
├── cdte_init *
├── gasb_init *
├── hgte_init *
└── insb_init *



In [ ]:
for path in structpath.fast_iter:
    print(path.get_node().get_ase())

In [ ]:
for path in workpath:
    print(path.get_node().is_finished_ok, path.key)

In [ ]:
def show_bands(worknode):
    from aiida_vasp.utils.sumo import get_sumo_bands_plotter
    plotter = get_sumo_bands_plotter(worknode.outputs.band_structure,
                                     structure=worknode.outputs.primitive_structure)
    return plotter

In [ ]:
node = workpath['gete_init'].get_node()

In [ ]:

from aiida.tools.visualization import Graph, pstate_node_styles


In [ ]:
g = Graph()

g.recurse_ancestors(load_node(613713))

g.graphviz

In [ ]:
get

In [ ]:
workpath = basepath['hse06_bs_soc']
for path in workpath:
    node = path.get_node()
    plotter = show_bands(node)
    gap = plotter.bs.get_band_gap()['energy']
    plotter.get_plot(title=node.label.split()[0] + f' Band Gap: {gap:.3f} eV SOC')

In [ ]:
workpath = basepath['hse06_bs']
for path in workpath:
    node = path.get_node()
    plotter = show_bands(node)
    gap = plotter.bs.get_band_gap()['energy']
    plotter.get_plot(title=node.label.split()[0] + f' Band Gap: {gap:.3f} eV')